In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Factory OEE & Downtime — a Beginner’s Guide (Synthetic, No Internet)

## Cell 1 — Changelog (Markdown)
Use this cell to track your changes version-to-version. Add one line per update so readers know what changed and you can compare CV → LB later.


In [2]:
# Repro
SEED = 1337
import os, random, math, datetime as dt
import numpy as np, pandas as pd
random.seed(SEED); np.random.seed(SEED)

# Paths
WORK = "./output"
os.makedirs(WORK, exist_ok=True)

print("SEED:", SEED)
print("Work dir:", WORK)


SEED: 1337
Work dir: ./output


## Cell 2 — Repro & Imports
Set a global **SEED** and import essentials.  
Why this matters: deterministic runs → easier debugging & apples-to-apples comparisons. Also make a `/kaggle/working` folder for outputs.


In [3]:
# Simulation horizon: DAYS days (parametro, ver mas abajo), per-minute resolution (fixed: use "min" not "T")  # MODIFICADO: comentario desactualizado del original (decia "3 days" pero DAYS=30)
DAYS = 30
FREQ = "min"  # minute frequency; future-proof

start = pd.Timestamp("2025-01-01 06:00:00")  # start at shift A
end = start + pd.Timedelta(days=DAYS)
time_index = pd.date_range(start, end, freq=FREQ, inclusive="left")

# 3 shifts of 8h covering 24h
def shift_name(ts):
    h = ts.hour
    if 6 <= h < 14:
        return "A"   # 06:00–14:00
    if 14 <= h < 22:
        return "B"   # 14:00–22:00
    return "C"      # 22:00–06:00

calendar = pd.DataFrame({
    "timestamp": time_index,
    "shift": [shift_name(ts) for ts in time_index]
})
calendar["day"] = calendar["timestamp"].dt.date

# Quick sanity check
print("Rows in calendar:", len(calendar))
print(calendar.head(3))


Rows in calendar: 43200
            timestamp shift         day
0 2025-01-01 06:00:00     A  2025-01-01
1 2025-01-01 06:01:00     A  2025-01-01
2 2025-01-01 06:02:00     A  2025-01-01


## Cell 3 — Settings & Shift Calendar
Create a **time index** (per-minute) for several days and map each timestamp to a **shift** (A/B/C) and **day**.  
This gives us a clean calendar to aggregate OEE by shift/day later.

In [4]:
# ==================== MODIFIED ====================
# Original Notebook has been configured to work with "machines", in This case is defined as a "Assembly" Line with 8 Stations.
# Each Station has a probability to fail.
# 'machine' is changed to Station ID (ST1..ST8)

STATION_SEQUENCE = [
    {"machine": "ST1", "station_id": "ST1", "station_type": "Robotics",       "process": "Product Introduction [1]"},
    {"machine": "ST2", "station_id": "ST2", "station_type": "CNC",            "process": "Mechanning Product 1"},
    {"machine": "ST3", "station_id": "ST3", "station_type": "Robotics",       "process": "Product Introduction [2]"},
    {"machine": "ST4", "station_id": "ST4", "station_type": "Assembly",       "process": "Assembly Products [1+2]"},
    {"machine": "ST5", "station_id": "ST5", "station_type": "Welding",        "process": "Join Products"},
    {"machine": "ST6", "station_id": "ST6", "station_type": "Cooling_System", "process": "Cooling"},
    {"machine": "ST7", "station_id": "ST7", "station_type": "Inspection",     "process": "Quality Control"},
    {"machine": "ST8", "station_id": "ST8", "station_type": "Packaging",      "process": "Packaging"},
]
MACHINE_TO_STATION = {s["machine"]: s for s in STATION_SEQUENCE}

MACHINES = [s["machine"] for s in STATION_SEQUENCE]

# Each ST has their own fail probability (Before, All Machines share same probability)
# fail_probs distinto por tipo de estacion (antes era un unico dict
# compartido por todas las maquinas). Cada estacion tiene una causa dominante
# Mechanical Cause and Electrical Cause, are triggered by data sensors.

# coherente con su proceso real (ver Markdown de la seccion).
# Nota: 'Starved' en Robotics = 0.0 (buffer deterministico, ver mas abajo).
# Nota: 'Mechanical' en Robotics/CNC = 0.0, 'Electrical' en Welding/
# Cooling_System = 0.0, y 'Quality' en Inspection = 0.0 -> estas causas YA
# NO son aleatorias: se disparan cuando su(s) sensor(es) asociado(s) cruzan
# un umbral tras una rampa de degradacion (ver STATION_SENSOR_DRIVER).
# Nota: 'Blocked'=0.0 en TODAS las estaciones, y 'Starved'=0.0 en las que
# no son Robotics -> ambas pasan a ser DETERMINISTAS, calculadas por una
# simulacion de flujo real con buffers entre estaciones (ver seccion
# AÑADIDO 'Flujo real / Blocked-Starved por buffer' mas abajo).
STATION_TYPE_FAIL_PROBS = {
    'Robotics':       {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0005,"Blocked":0.0,"Starved":0.0,"Quality":0.0005},
    'CNC':            {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0005,"Blocked":0.0,"Starved":0.0,"Quality":0.0005},
    'Assembly':       {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0005,"Blocked":0.0,"Starved":0.0,"Quality":0.0005},
    'Welding':        {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0005,"Blocked":0.0,"Starved":0.0,"Quality":0.0005},
    'Cooling_System': {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0004,"Blocked":0.0,"Starved":0.0,"Quality":0.0005},
    'Inspection':     {"Mechanical":0.0004,"Electrical":0.0,   "Changeover":0.0005,"Blocked":0.0,"Starved":0.0,"Quality":0.0},
    'Packaging':      {"Mechanical":0.0,"Electrical":0.0,   "Changeover":0.0016,"Blocked":0.0,"Starved":0.0,"Quality":0.0004},
}

# AÑADIDO: mapa causal sensor -> fallo. MODIFICADO: cada station_type puede
# tener VARIOS drivers independientes a la vez (lista), uno por causa. Regla
# general: 'vibration'+'air_pressure' -> Mechanical, en TODA estacion que
# tenga ambos sensores presentes; 'temperature'+'energy_consumption'+
# 'humidity' -> Electrical, en TODA estacion que tenga los tres presentes.
# CNC y Welding tienen los 5 sensores -> les aplican AMBAS reglas a la vez
# (Mechanical Y Electrical, cada una con su propio ciclo de rampa).
# AÑADIDO: ritmo de degradacion propio por sensor (equivalente a una
# 'probabilidad de fallo' propia, pero expresada como frecuencia temporal:
# cada cuantos minutos, de media, arranca un nuevo ciclo de degradacion).
# Valores mas bajos = falla mas a menudo. Basado en frecuencias tipicas de
# fallo industrial: desgaste mecanico (vibration) es de lo mas comun; fugas
# neumaticas (air_pressure) y entrada de humedad (humidity) son mas raras.
SENSOR_FAILURE_RATE = {
    "vibration":          (250, 550),
    "air_pressure":       (400, 900),
    "temperature":        (300, 700),
    "energy_consumption": (350, 800),
    "humidity":           (500, 1000),
}

STATION_SENSOR_DRIVER = {
    'Robotics':       [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "vibration",    "baseline": 49.18, "threshold": 58.51},
            {"sensor": "air_pressure", "baseline": 5.0,   "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "temperature",       "baseline": 73.61, "threshold": 89.35},
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
        ]},
    ],
    'CNC':            [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "vibration",    "baseline": 49.18, "threshold": 58.51},
            {"sensor": "air_pressure", "baseline": 5.0,   "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "temperature",       "baseline": 73.61, "threshold": 89.35},
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
            {"sensor": "humidity",          "baseline": 55.0,  "threshold": 75.0},
        ]},
    ],
    'Assembly':       [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "vibration",    "baseline": 49.18, "threshold": 58.51},
            {"sensor": "air_pressure", "baseline": 5.0,   "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
        ]},
    ],
    'Welding':        [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "vibration",    "baseline": 49.18, "threshold": 58.51},
            {"sensor": "air_pressure", "baseline": 5.0,   "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "temperature",       "baseline": 73.61, "threshold": 89.35},
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
            {"sensor": "humidity",          "baseline": 55.0,  "threshold": 75.0},
        ]},
    ],
    'Cooling_System': [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "air_pressure", "baseline": 5.0, "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "temperature",       "baseline": 73.61, "threshold": 89.35},
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
            {"sensor": "humidity",          "baseline": 55.0,  "threshold": 75.0},
        ]},
    ],
    'Packaging':      [
        {"cause": "Mechanical", "sensors": [
            {"sensor": "vibration",    "baseline": 49.18, "threshold": 58.51},
            {"sensor": "air_pressure", "baseline": 5.0,   "threshold": 3.2},
        ]},
        {"cause": "Electrical", "sensors": [
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
        ]},
    ],
    'Inspection':     [
        {"cause": "Electrical", "sensors": [
            {"sensor": "energy_consumption","baseline": 2.75,  "threshold": 4.5},
        ]},
    ],
}
# =================== FIN AÑADIDO ===================

# ==================== AÑADIDO (rediseño monitoreo continuo) ====================
def generate_continuous_sensor_series(n_minutes, baseline, threshold,
                                       healthy_gap=(300, 700), ramp_len=(60, 240),
                                       down_len=(5, 40)):
    """MODIFICADO: reemplaza el mecanismo anterior (rampa programada de
    antemano, disparo SIEMPRE en el minuto final aunque el ruido de fondo
    cruzara el umbral antes por casualidad -> inconsistente).

    Ahora el sensor es un proceso CONTINUO minuto a minuto:
    - Fondo (funcionamiento normal): random walk con reversion a la media,
      RECORTADO (clip) para que nunca pueda alcanzar el umbral por azar
      (max_bg_dev = 60% de la distancia baseline->umbral). Asi el ruido de
      fondo jamas cruza el umbral -> ya no hay 'falsos cruces' sin disparo.
    - Rampa: tras un periodo sano, sube/baja con curva convexa hacia el
      umbral. El disparo ocurre en el PRIMER minuto real en que el valor
      cruza el umbral (monitoreo continuo autentico), no en un instante
      pre-calculado.
    - Post-fallo: el valor se mantiene cerca del umbral (con ruido) durante
      la duracion del paro, luego vuelve al fondo normal.

    Devuelve (values, triggers) donde triggers es una lista de
    (minuto_disparo, duracion_paro_min)."""
    rising = threshold > baseline
    amplitude = abs(threshold - baseline)
    bg_noise_std = amplitude * 0.15
    max_bg_dev = amplitude * 0.6
    direction = 1 if rising else -1

    values = np.empty(n_minutes)
    triggers = []
    walk = 0.0
    in_ramp = False
    in_downtime = False
    downtime_remaining = 0
    onset_countdown = np.random.randint(healthy_gap[0], healthy_gap[1])
    ramp_elapsed = 0
    ramp_duration = 0
    power = 2.0

    for t in range(n_minutes):
        if in_downtime:
            values[t] = threshold + np.random.normal(0, amplitude * 0.08)
            downtime_remaining -= 1
            if downtime_remaining <= 0:
                in_downtime = False
                walk = 0.0
                onset_countdown = np.random.randint(healthy_gap[0], healthy_gap[1])
            continue
        if not in_ramp:
            walk = walk * 0.85 + np.random.normal(0, bg_noise_std)
            walk = np.clip(walk, -max_bg_dev, max_bg_dev)
            values[t] = baseline + walk
            onset_countdown -= 1
            if onset_countdown <= 0:
                in_ramp = True
                ramp_duration = np.random.randint(ramp_len[0], ramp_len[1])
                ramp_elapsed = 0
                power = np.random.uniform(1.5, 3.0)
            continue
        # dentro de la rampa: curva convexa hacia el umbral
        ramp_elapsed += 1
        progress = min(1.0, ramp_elapsed / ramp_duration)
        target = baseline + direction * amplitude * (progress ** power)
        noise = np.random.normal(0, amplitude * 0.05)
        values[t] = target + noise
        crossed = (values[t] >= threshold) if rising else (values[t] <= threshold)
        if crossed or ramp_elapsed >= ramp_duration:
            dlen = np.random.randint(down_len[0], down_len[1])
            triggers.append((t, dlen))
            in_ramp = False
            in_downtime = True
            downtime_remaining = dlen
    return values, triggers
# =================== FIN AÑADIDO ===================

DOWNTIME_CAUSES = [
    "Mechanical", "Electrical", "Changeover", "Blocked", "Starved", "Quality"
]

# ==================== AÑADIDO ====================
def generate_starved_blocks(n_minutes, autonomy=30, mean_gap=10, rng=None):
    """Solo se usa en estaciones Robotics (ST1/ST3, las que introducen producto
    en la linea). Simula visitas de reposicion del operario con intervalos
    irregulares (media 'mean_gap' min). Si el hueco entre visitas supera la
    'autonomy' (30 min de buffer), la maquina se queda sin material -> Starved,
    de forma DETERMINISTA (no probabilistica como el resto de causas)."""
    rng = rng or np.random.default_rng()
    mask = np.zeros(n_minutes, dtype=bool)
    t = 0
    while t < n_minutes:
        gap = max(1, int(rng.exponential(mean_gap)))
        visit = min(t + gap, n_minutes)
        if gap > autonomy:
            mask[t + autonomy:visit] = True
        t = visit
    return mask
# =================== FIN AÑADIDO ===================

# MODIFICADO: firma original era
#   def synth_machine(machine, base_rate=6, scrap_rate=0.02, fail_probs={...}, down_len=(5,40)):
# ahora fail_probs se pasa por estacion, se añade autonomy_buffer (Starved
# deterministico) y sensor_driver (Mechanical/Electrical deterministico por
# rampa de sensor -> ver bloque AÑADIDO dentro de la funcion).
def synth_machine(machine, fail_probs, base_rate=6, scrap_rate=0.02,
                  down_len=(5, 40), autonomy_buffer=None, sensor_driver=None, quality_burst=False):
    """
    Return per-minute time series for one machine:
    - is_running
    - units (actual output)
    - cause markers
    - scrap
    """
    df = pd.DataFrame({"timestamp": time_index})
    df["machine"] = machine  # se sobreescribe despues con el nombre real de la linea (ver bucle LINES x MACHINES)
    df["is_running"] = 1
    for c in DOWNTIME_CAUSES:
        df[f"cause_{c}"] = 0
    df["trigger_sensor"] = ""  # AÑADIDO: que sensor concreto disparo el evento (dentro de una misma 'cause' puede haber varios sensores posibles)

    n = len(df)

    # ==================== AÑADIDO (rediseño monitoreo continuo) ====================
    # MODIFICADO: se sustituye el mecanismo anterior (rampa programada de
    # antemano con disparo en un instante fijo) por una simulacion CONTINUA
    # por sensor (ver generate_continuous_sensor_series): el ruido de fondo
    # esta acotado para que NUNCA cruce el umbral por azar, y el disparo
    # ocurre en el primer minuto REAL en que el sensor cruza su umbral
    # (monitoreo autentico, no un guion pre-calculado).
    flat_targets = [(drv["cause"], spec) for drv in (sensor_driver or []) for spec in drv["sensors"]]
    all_events = []  # (minuto_disparo, duracion_paro, cause, sensor_name)
    for (cause, spec) in flat_targets:
        gap = SENSOR_FAILURE_RATE.get(spec["sensor"], (300, 700))  # ritmo propio por sensor
        values, triggers = generate_continuous_sensor_series(
            n, spec["baseline"], spec["threshold"], healthy_gap=gap, down_len=down_len
        )
        df[spec["sensor"]] = values  # AÑADIDO: la serie completa (fondo+rampa+post-fallo) ya queda asignada aqui
        for (t, dlen) in triggers:
            all_events.append((t, dlen, cause, spec["sensor"]))

    # Orden cronologico real cruzando todos los sensores: el que dispara
    # antes en el tiempo gana, sin importar el orden de la lista de sensores.
    all_events.sort(key=lambda ev: ev[0])
    for (t, dlen, cause, sensor_name) in all_events:
        if t < n and df.loc[t, "is_running"] == 1:
            j = min(t + dlen, n)
            df.loc[t:j, "is_running"] = 0
            df.loc[t, f"cause_{cause}"] = 1
            df.loc[t, "trigger_sensor"] = sensor_name
    # =================== FIN AÑADIDO ===================

    # introduce random downtime blocks by cause (causas SIN sensor asociado;
    # las que si tienen sensor_driver ya vienen con fail_probs=0.0, asi que
    # este bucle no las vuelve a disparar por azar)
    i = 0
    while i < n:
        # independent cause trigger probabilities
        cause_triggers = [(c, np.random.rand() < fail_probs[c]) for c in DOWNTIME_CAUSES]
        fired = [c for c, hit in cause_triggers if hit]
        if fired and df.loc[i, "is_running"] == 1:
            cause = np.random.choice(fired)
            L = np.random.randint(down_len[0], down_len[1])  # minutes down
            j = min(i+L, n)
            df.loc[i:j, "is_running"] = 0
            df.loc[i, f"cause_{cause}"] = 1  # mark start
            i = j
        else:
            i += 1

    # ================== AÑADIDO ==================
    # Starved deterministico por autonomia de buffer (solo Robotics)
    if autonomy_buffer is not None:
        starved_mask = generate_starved_blocks(n, autonomy=autonomy_buffer, mean_gap=8)  # MODIFICADO: antes 10, ahora el operario visita mas a menudo (~2.5% starved en vez de ~4.9%)
        apply_mask = starved_mask & (df["is_running"].values == 1)
        block_starts = np.where(apply_mask & ~np.r_[False, apply_mask[:-1]])[0]
        for s in block_starts:
            e = s
            while e < n and starved_mask[e]:
                e += 1
            df.loc[s:e-1, "is_running"] = 0
            df.loc[s, "cause_Starved"] = 1
    # ================ FIN AÑADIDO =================

    # production: Poisson when running, else 0; small shift-based rate drift
    rate_shift = df["timestamp"].dt.hour.map(lambda h: 0.5 if h in [6,14,22] else 0.0)  # slightly higher at shift start
    lam = base_rate + rate_shift.values
    df["units"] = np.where(df["is_running"]==1, np.random.poisson(lam), 0)

    # scrap generation
    df["scrap"] = (np.random.rand(n) < scrap_rate).astype(int) * (df["units"]>0).astype(int)

    # ==================== AÑADIDO ====================
    # 'Brotes' de scrap (solo si quality_burst=True, usado en Inspection):
    # un defecto sistematico real viene en RACHAS (p.ej. una herramienta
    # aguas arriba se desajusta y produce varias piezas malas seguidas), no
    # aislado al azar. Con scrap_rate=2% independiente, 3 seguidas es casi
    # imposible (0.02^3 ~ 0.0008%) -> se inyectan ventanas cortas (3-8 min)
    # de scrap_rate alto (70%) de forma periodica, que es lo que dispara la
    # regla de Quality (3 consecutivas) mas abajo.
    if quality_burst:
        t = 0
        while t < n:
            gap = np.random.randint(300, 900)
            onset = t + gap
            if onset >= n:
                break
            burst_len = np.random.randint(3, 8)
            end = min(onset + burst_len, n)
            burst_scrap = (np.random.rand(end - onset) < 0.7).astype(int)
            df.loc[onset:end - 1, "scrap"] = burst_scrap * (df.loc[onset:end - 1, "units"] > 0).astype(int)
            t = end
    # =================== FIN AÑADIDO ===================

    df["good_units"] = df["units"] - df["scrap"]

    # ==================== AÑADIDO ====================
    # Metadatos de estacion (antes no existian: solo habia 'machine')
    st = MACHINE_TO_STATION[machine]
    df["station_id"] = st["station_id"]
    df["station_type"] = st["station_type"]
    df["process"] = st["process"]
    df["sequence_order"] = int(st["station_id"][2:])
    # =================== FIN AÑADIDO ===================

    return df

# ==================== AÑADIDO ====================
# Numero de LINEAS de produccion a simular. Cambia N_LINES para tener mas
# de una linea (M1, M2, M3...) — CADA linea siempre incluye las 8
# estaciones completas (ST1..ST8), con la misma logica de causas/sensores.
N_LINES = 1
LINES = [f"M{i+1}" for i in range(N_LINES)]
# =================== FIN AÑADIDO ===================

# MODIFICADO: antes era
#   raw = pd.concat([synth_machine(m) for m in MACHINES], ignore_index=True)
# (una sola linea implicita). Ahora se itera LINEAS x ESTACIONES: cada linea
# genera sus propias 8 estaciones de forma independiente (misma logica de
# fail_probs/autonomy_buffer/sensor_driver por station_type), y 'machine'
# se sobreescribe con el nombre real de la linea tras generar cada estacion.
frames = []
for line in LINES:
    for m in MACHINES:
        st = MACHINE_TO_STATION[m]
        autonomy = 30 if st["station_type"] == "Robotics" else None
        driver = STATION_SENSOR_DRIVER.get(st["station_type"])
        burst = st["station_type"] == "Inspection"  # AÑADIDO: brotes de scrap solo en Inspection (para la regla de 3 consecutivas)
        df_st = synth_machine(m, fail_probs=STATION_TYPE_FAIL_PROBS[st["station_type"]],
                               autonomy_buffer=autonomy, sensor_driver=driver, quality_burst=burst)
        df_st["machine"] = line  # AÑADIDO: nombre real de la linea (permite N_LINES > 1)
        frames.append(df_st)
raw = pd.concat(frames, ignore_index=True)
raw = raw.merge(calendar, on="timestamp", how="left")
raw = raw.sort_values(["machine", "sequence_order", "timestamp"]).reset_index(drop=True)  # MODIFICADO: orden por linea + sequence_order

# ==================== AÑADIDO ====================
# Detalle semantico: Starved en estaciones Robotics = falta de reposicion del
# operario (no es un fallo mecanico de la maquina en si).
raw["stop_reason_detail"] = ""
mask_operator_starved = (raw["station_type"] == "Robotics") & (raw["cause_Starved"] == 1)
raw.loc[mask_operator_starved, "stop_reason_detail"] = "Operator supply gap (infeed not replenished)"
# =================== FIN AÑADIDO ===================

print("Rows:", len(raw))
raw.head()


Rows: 345600


,timestamp,machine,is_running,cause_Mechanical,cause_Electrical,cause_Changeover,cause_Blocked,cause_Starved,cause_Quality,trigger_sensor,...,scrap,good_units,station_id,station_type,process,sequence_order,humidity,shift,day,stop_reason_detail
0,2025-01-01 06:00:00,M1,1,0,0,0,0,0,0,,...,0,6,ST1,Robotics,Product Introduction [1],1,NaN,A,2025-01-01,
1,2025-01-01 06:01:00,M1,1,0,0,0,0,0,0,,...,0,8,ST1,Robotics,Product Introduction [1],1,NaN,A,2025-01-01,
2,2025-01-01 06:02:00,M1,1,0,0,0,0,0,0,,...,0,7,ST1,Robotics,Product Introduction [1],1,NaN,A,2025-01-01,
3,2025-01-01 06:03:00,M1,1,0,0,0,0,0,0,,...,0,7,ST1,Robotics,Product Introduction [1],1,NaN,A,2025-01-01,
4,2025-01-01 06:04:00,M1,1,0,0,0,0,0,0,,...,0,6,ST1,Robotics,Product Introduction [1],1,NaN,A,2025-01-01,


## Cell 4 — Synthetic Line & Downtime Generator
Simulate **8 stations in sequence** (ST1→ST8) *(AÑADIDO: antes eran 2 máquinas idénticas M1/M2)*:
- Cada estación tiene un `fail_probs` propio, con una **causa de fallo dominante coherente con su proceso** (ej. CNC → Mechanical, Inspection → Quality, Welding → Electrical). *(AÑADIDO)*
- **Starved en Robotics (ST1/ST3) ya no es aleatorio**: se modela con una autonomía de buffer de 30 min — si el operario tarda más en reponer material, la máquina se para de forma determinista. *(AÑADIDO)*
- Random **downtime blocks** con causas etiquetadas (lógica original conservada).
- **Units/min** solo si está corriendo; pequeño repunte al inicio de turno (original).
- **Scrap** para habilitar los cálculos de Quality (original).

Salida: tabla con `timestamp, machine, station_id, station_type, process, sequence_order, is_running, units, scrap, good_units, shift, day, stop_reason_detail`.

## AÑADIDO — Enriquecimiento con sensores (Temperature, Vibration, Humidity, Pressure, Energy_Consumption)

Sección nueva. Usa `smart_manufacturing_data.csv` (dataset real de Kaggle) como fuente de valores realistas.

**Monitoreo continuo real (no un guion pre-calculado)**: cada sensor driver es un proceso CONTINUO minuto a minuto (`generate_continuous_sensor_series`). El ruido de fondo esta acotado (nunca puede alcanzar el umbral por azar); tras un periodo sano, el sensor sube/baja con curva convexa, y el disparo ocurre en el PRIMER minuto real en que cruza su umbral -> ya no hay falsos cruces sin disparo ni disparos en un instante fijo independientes del valor real.

Para el resto de causas/estaciones (Quality, Assembly, Packaging, Inspection...) no existe un sensor con señal real diferenciable en el dataset externo (`pressure`/`humidity`/`energy_consumption` no distinguen anomalía de normalidad), así que ahí se mantiene el muestreo post-hoc: fallo real de máquina → `anomaly_flag==1`; resto → `anomaly_flag==0`.

**Sensores instalados por estación**: no todas las estaciones tienen los 5 sensores físicamente instalados — se deja `NaN` donde no aplica (`STATION_SENSORS_PRESENT`), respetando siempre el sensor causal de cada estación.

In [5]:
# ==================== AÑADIDO (sensores) ====================
sensor_source = pd.read_csv("smart_manufacturing_data.csv")
# MODIFICADO: se renombra 'pressure' -> 'air_pressure'. El dataset externo no
# documenta unidades; se interpreta como presion de aire (no de proceso
# hidraulico/neumatico), y por eso pasa a tener sentido tambien en Robotics
# (presion de aire del sistema neumatico de agarre/pinza del brazo robotico).
sensor_source = sensor_source.rename(columns={"pressure": "air_pressure"})
SENSOR_COLS = ["temperature", "vibration", "humidity", "air_pressure", "energy_consumption"]

anomaly_pool = sensor_source[sensor_source["anomaly_flag"] == 1][SENSOR_COLS].reset_index(drop=True)
normal_pool  = sensor_source[sensor_source["anomaly_flag"] == 0][SENSOR_COLS].reset_index(drop=True)

# AÑADIDO: mapa estacion -> sensores que son DRIVER ahi (ya simulados de
# forma continua dentro de synth_machine, NO tocar). El resto de sensores
# presentes en cada estacion (ej. humidity en Packaging, que esta presente
# pero no es driver ahi) se rellenan aqui con el muestreo post-hoc de fondo.
DRIVER_SENSOR_MAP = {
    st: {spec["sensor"] for drv in driver_list for spec in drv["sensors"]}
    for st, driver_list in STATION_SENSOR_DRIVER.items()
}

# Causa activa por minuto (propagada dentro de cada bloque is_running==0,
# ya que cause_X solo se marca en el minuto de INICIO del bloque)
raw["active_cause"] = ""
for c in DOWNTIME_CAUSES:
    raw.loc[raw[f"cause_{c}"] == 1, "active_cause"] = c
block_id = (raw["is_running"] != raw.groupby("machine")["is_running"].shift(1)).cumsum()
raw["active_cause"] = raw.groupby(block_id)["active_cause"].transform(lambda s: s.replace("", np.nan).ffill().fillna(""))
raw.loc[raw["is_running"] == 1, "active_cause"] = ""

# MODIFICADO (rediseño monitoreo continuo): antes esto sobreescribia TODOS
# los sensores de TODAS las filas con muestreo post-hoc (incluidos los
# sensores driver, que ya venian correctamente simulados de forma continua
# desde synth_machine -> se perdian). Ahora solo se rellenan aqui los
# sensores que NO son driver en su estacion (fondo/ruido de contexto, sin
# relacion causal con ningun fallo, ej. humidity en Packaging).
MACHINE_FAULT_CAUSES = {"Mechanical", "Electrical", "Quality"}
is_fault = raw["active_cause"].isin(MACHINE_FAULT_CAUSES)

rng = np.random.default_rng(SEED)
for col in SENSOR_COLS:
    is_driver_here = raw["station_type"].map(lambda st: col in DRIVER_SENSOR_MAP.get(st, set()))
    fillable = ~is_driver_here  # solo donde este sensor NO es driver en esa estacion
    if col not in raw.columns:
        raw[col] = np.nan
    fault_mask = fillable & is_fault
    normal_mask = fillable & ~is_fault
    if fault_mask.sum() > 0:
        idx = rng.integers(0, len(anomaly_pool), size=fault_mask.sum())
        raw.loc[fault_mask, col] = anomaly_pool[col].values[idx]
    if normal_mask.sum() > 0:
        idx = rng.integers(0, len(normal_pool), size=normal_mask.sum())
        raw.loc[normal_mask, col] = normal_pool[col].values[idx]

raw = raw.drop(columns=["active_cause"])

# ==================== AÑADIDO ====================
# Fix de calidad de datos: 'smart_manufacturing_data.csv' (fuente externa)
# contiene 37 filas con vibration NEGATIVA (ruido de sensor mal calibrado en
# el dataset real de origen) -> fisicamente imposible (la vibracion es una
# amplitud/RMS, nunca negativa). Se recorta a un minimo de 0 tras el
# muestreo, ya que heredamos ese ruido del dataset externo al copiarlo.
raw["vibration"] = raw["vibration"].clip(lower=0)
# =================== FIN AÑADIDO ===================

# ==================== AÑADIDO ====================
# Paso 3: no todas las estaciones tienen instalados fisicamente los 5
# sensores. Se deja NaN en las combinaciones estacion-sensor que no aplican
# (sensor no instalado = dato ausente, no cero). Los sensores CAUSALES de
# cada estacion (ver STATION_SENSOR_DRIVER) siempre quedan incluidos aqui.
STATION_SENSORS_PRESENT = {
    'Robotics':       ['temperature', 'vibration', 'air_pressure', 'energy_consumption'],  # MODIFICADO: se añade air_pressure (pinza neumatica)
    'CNC':            ['temperature', 'vibration', 'humidity', 'air_pressure', 'energy_consumption'],
    'Assembly':       ['vibration', 'air_pressure', 'energy_consumption'],
    'Welding':        ['temperature', 'vibration', 'humidity', 'air_pressure', 'energy_consumption'],
    'Cooling_System': ['temperature', 'humidity', 'air_pressure', 'energy_consumption'],  # MODIFICADO: se añade air_pressure
    'Inspection':     ['energy_consumption'],
    'Packaging':      ['vibration', 'humidity', 'air_pressure', 'energy_consumption'],
}
for station_type, sensors_present in STATION_SENSORS_PRESENT.items():
    absent = [c for c in SENSOR_COLS if c not in sensors_present]
    mask = raw["station_type"] == station_type
    raw.loc[mask, absent] = np.nan
# =================== FIN AÑADIDO ===================

raw[SENSOR_COLS].describe()
# =================== FIN AÑADIDO (sensores) ===================


,temperature,vibration,humidity,air_pressure,energy_consumption
count,216000.000000,259200.000000,172800.000000,302400.000000,345600.000000
mean,75.146406,50.265691,56.075938,4.852584,2.912066
std,5.379139,3.272212,9.152296,0.595310,0.591322
min,64.166000,43.582000,30.000000,2.583173,1.700000
25%,71.678961,48.171638,51.027404,4.536045,2.526945
50%,74.596022,49.870103,55.880120,4.903096,2.854123
75%,77.910978,51.927568,60.909064,5.241724,3.220954
max,94.261174,61.941455,80.961089,6.080000,5.049513


## AÑADIDO — Flujo real entre estaciones: Blocked / Starved por buffer

`Blocked` y `Starved` (salvo el Starved-operario de Robotics, que no cambia) dejan de ser aleatorios: se simula un **flujo de piezas real** entre estaciones, con un tiempo de ciclo (`cycle_time`, base ~9.2 s/pieza — margen bajo los 10 s/pieza nominales de `base_rate=6`) que varía con el tiempo, y buffers de 5 piezas entre cada par de estaciones conectadas.

**Topología de la línea** (ST1 y ST3 son dos entradas de material independientes que convergen en ST4):

```
ST1 → ST2 ─┐
           ├─→ ST4 → ST5 → ST6 → ST7 → ST8
      ST3 ─┘
```

- **Blocked**: la estación no puede sacar una pieza porque el buffer hacia la siguiente estación está lleno
- **Starved**: la estación no tiene pieza que procesar porque el buffer de la estación anterior está vacío
- **ST1** (introduce producto 1) no puede estar Starved por flujo (no tiene buffer de entrada, es puro infeed) — usa el Starved-operario (autonomía de 30 min)
- **ST3** (introduce producto 2, uniéndolo al producto 1 que llega de ST2) puede estar Starved por DOS motivos: buffer de ST2 vacío (falta producto 1) O autonomía de operario agotada (falta producto 2)
- ST8 no puede estar Blocked (fin de línea)

In [6]:
# ==================== AÑADIDO (flujo real) ====================
# MODIFICADO: cadena LINEAL (antes ST2 y ST3 confluian en paralelo hacia
# ST4). Ahora cada estacion depende SOLO de la inmediatamente anterior:
# ST1 introduce producto 1 -> ST2 lo procesa -> ST3 lo recibe Y ADEMAS
# introduce producto 2 (autonomia de operario, mecanismo ya existente para
# estaciones Robotics) -> ST4 los une -> ST5 suelda -> ST6 -> ST7 -> ST8.
FLOW_TOPOLOGY = {
    "ST1": {"in": [],        "out": "B_1_2"},
    "ST2": {"in": ["B_1_2"], "out": "B_2_3"},
    "ST3": {"in": ["B_2_3"], "out": "B_3_4"},
    "ST4": {"in": ["B_3_4"], "out": "B_4_5"},
    "ST5": {"in": ["B_4_5"], "out": "B_5_6"},
    "ST6": {"in": ["B_5_6"], "out": "B_6_7"},
    "ST7": {"in": ["B_6_7"], "out": "B_7_8"},
    "ST8": {"in": ["B_7_8"], "out": None},
}
BUFFER_CAPACITY = 15  # piezas (subido desde 5: con cycle_time~9.2s cada estacion produce ~6 piezas/min, asi que un buffer de 5 se llenaba/vaciaba en menos de 1 minuto y generaba Blocked/Starved excesivo ~33-38%. Con 15 hay margen de ~2.5 min de absorcion)
STATION_ORDER = ["ST1", "ST2", "ST3", "ST4", "ST5", "ST6", "ST7", "ST8"]

def generate_cycle_time(n_minutes, base=9.2, noise_sd=0.3, slow_gap=(400, 800),
                         slow_len=(30, 120), slow_extra=(1.5, 4.0)):
    """cycle_time (s/pieza): ruido pequeño sobre 'base' + episodios de
    ralentizacion (subidas temporales) que rompen el margen de holgura y
    generan Blocked/Starved reales en el resto de la linea."""
    ct = base + np.random.normal(0, noise_sd, n_minutes)
    t = 0
    while t < n_minutes:
        gap = np.random.randint(slow_gap[0], slow_gap[1])
        onset = t + gap
        if onset >= n_minutes:
            break
        L = np.random.randint(slow_len[0], slow_len[1])
        end = min(onset + L, n_minutes)
        extra = np.random.uniform(slow_extra[0], slow_extra[1])
        ct[onset:end] += extra
        t = end
    return np.clip(ct, 5.0, None)

n_min = len(time_index)

for line in LINES:
    line_mask = raw["machine"] == line
    # cycle_time por estacion (indexado 0..n_min-1, alineado con time_index)
    cycle_time = {st: generate_cycle_time(n_min) for st in STATION_ORDER}

    # vista rapida por estacion: is_running/cause actuales (ya generados por
    # Mechanical/Electrical/Quality/Changeover/Starved-operario), y arrays
    # de units/is_running/cause_Blocked/cause_Starved a actualizar in-place
    st_data = {}
    for st in STATION_ORDER:
        idx = raw.index[line_mask & (raw["station_id"] == st)]
        idx = idx[np.argsort(raw.loc[idx, "timestamp"].values)]
        st_data[st] = {
            "idx": idx.to_numpy(),
            "is_running": raw.loc[idx, "is_running"].to_numpy().copy(),
            "units": raw.loc[idx, "units"].to_numpy().copy(),
            "cause_Blocked": raw.loc[idx, "cause_Blocked"].to_numpy().copy(),
            "cause_Starved": raw.loc[idx, "cause_Starved"].to_numpy().copy(),
        }

    # MODIFICADO: buffers arrancan a MEDIA capacidad (antes llenos del todo),
    # ya que arrancar llenos bloqueaba en cascada las estaciones de entrada
    # desde el minuto 0 (no habia espacio para empujar nada). Media capacidad
    # simula una linea que ya llevaba un rato en marcha antes de empezar a medir.
    buffers = {b: BUFFER_CAPACITY // 2 for b in ["B_1_2", "B_2_3", "B_3_4", "B_4_5", "B_5_6", "B_6_7", "B_7_8"]}

    # AÑADIDO: acumulador de fase por estacion. int(60/cycle_time) trunca y
    # pierde sistematicamente la parte fraccionaria cada minuto (con
    # cycle_time~9.2s, 60/9.2=6.52 -> floor()=6, un 8% de capacidad perdida
    # de forma constante, no aleatoria) -> eso desestabilizaba TODA la cadena
    # de buffers. Se acumula la fraccion sobrante y se libera una pieza extra
    # cuando el acumulado llega a 1, para que la capacidad media real sea
    # exactamente 60/cycle_time, sin sesgo.
    phase = {st: 0.0 for st in STATION_ORDER}

    for t in range(n_min):
        for st in STATION_ORDER:
            d = st_data[st]
            topo = FLOW_TOPOLOGY[st]
            already_down = d["is_running"][t] == 0  # por otra causa (Mechanical/Electrical/Quality/Changeover/Starved-operario)
            if already_down:
                continue

            # AÑADIDO (fix): la causa solo se marca en el minuto de INICIO
            # del bloqueo/desabastecimiento (igual que el resto del pipeline:
            # cause_Mechanical, cause_Electrical, etc. tambien solo marcan el
            # inicio). Antes se marcaba en TODOS los minutos del bloqueo, lo
            # que generaba en 'downtime_events' un evento SOLAPADO por cada
            # minuto del mismo bloqueo (ej. un bloqueo de 4 min generaba 4
            # filas de evento en vez de 1), inflando el Pareto y cualquier
            # analisis de duracion. 'was_running_prev' mira el minuto anterior
            # de la MISMA estacion (t-1) para saber si este es un minuto de
            # inicio o la continuacion de un bloqueo ya empezado.
            was_running_prev = (t == 0) or (d["is_running"][t - 1] == 1)

            out_buf = topo["out"]
            if out_buf is not None and buffers[out_buf] >= BUFFER_CAPACITY:
                d["is_running"][t] = 0
                if was_running_prev:
                    d["cause_Blocked"][t] = 1
                d["units"][t] = 0
                continue

            in_bufs = topo["in"]
            if in_bufs and any(buffers[b] <= 0 for b in in_bufs):
                d["is_running"][t] = 0
                if was_running_prev:
                    d["cause_Starved"][t] = 1
                d["units"][t] = 0
                continue

            phase[st] += 60.0 / cycle_time[st][t]
            max_by_pace = int(phase[st])
            phase[st] -= max_by_pace
            avail_in = min([buffers[b] for b in in_bufs], default=max_by_pace)
            space_out = (BUFFER_CAPACITY - buffers[out_buf]) if out_buf is not None else max_by_pace
            output = max(0, min(max_by_pace, avail_in, space_out))

            for b in in_bufs:
                buffers[b] -= output
            if out_buf is not None:
                buffers[out_buf] += output
            d["units"][t] = output

    for st in STATION_ORDER:
        d = st_data[st]
        raw.loc[d["idx"], "is_running"] = d["is_running"]
        raw.loc[d["idx"], "units"] = d["units"]
        raw.loc[d["idx"], "cause_Blocked"] = d["cause_Blocked"]
        raw.loc[d["idx"], "cause_Starved"] = d["cause_Starved"]

# 'good_units' vuelve a depender de 'units' (scrap se conserva tal cual, salvo
# que ahora no puede superar las nuevas 'units' por estacion)
raw["scrap"] = np.minimum(raw["scrap"], raw["units"])
raw["good_units"] = raw["units"] - raw["scrap"]

print(raw.groupby("station_id")[["cause_Blocked", "cause_Starved"]].apply(lambda g: (g==1).sum()))
# =================== FIN AÑADIDO (flujo real) ===================


            cause_Blocked  cause_Starved
station_id                              
ST1                   903            116
ST2                   771            244
ST3                   609            515
ST4                   526            589
ST5                   378            660
ST6                   236            806
ST7                   183            894
ST8                     0            948


## AÑADIDO — Regla determinista de Quality en Inspection: 3 scrap consecutivos

Inspection (ST7) se queda sin sensor causal propio (solo tiene `energy_consumption`, sin `temperature`/`humidity` para completar el driver de Electrical). En su lugar, `Quality` se dispara con una **regla tipo SPC** (similar a las reglas de Western Electric/Nelson que ya usa el control X̄-R): si la estación produce **3 piezas de scrap seguidas** mientras está corriendo, se para por Quality — un patrón de calidad real (defecto sistemático), no una etiqueta aleatoria.

In [7]:
# ==================== AÑADIDO (regla Quality Inspection) ====================
def apply_quality_scrap_rule(is_running, scrap, units, cause_Quality, trigger_sensor, down_len=(5, 40), n_consecutive=3):
    """3 piezas de scrap seguidas (mientras is_running==1) -> paro por Quality.
    Arrays modificados IN-PLACE. Se reinicia el contador al parar (por
    cualquier causa) o al producir una pieza buena."""
    n = len(is_running)
    consec = 0
    t = 0
    while t < n:
        if is_running[t] == 1:
            consec = consec + 1 if scrap[t] == 1 else 0
            if consec >= n_consecutive:
                start = min(t + 1, n - 1)
                # AÑADIDO (fix): si 'start' ya esta parada por otra causa
                # (ej. Starved del flujo de buffers, que corre ANTES que
                # esta regla), no forzar el paro encima -> evitaba que un
                # mismo minuto tuviera 2 causas a la vez (Quality Y Starved).
                if is_running[start] != 1:
                    consec = 0
                    t += 1
                    continue
                L = np.random.randint(down_len[0], down_len[1])
                end = min(start + L, n)
                is_running[start:end] = 0
                units[start:end] = 0
                scrap[start:end] = 0
                cause_Quality[start] = 1
                trigger_sensor[start] = "scrap_rule"  # AÑADIDO: no viene de un sensor fisico, sino de la regla de 3 consecutivas
                consec = 0
                t = end
                continue
        else:
            consec = 0
        t += 1

for line in LINES:
    idx = raw.index[(raw["machine"] == line) & (raw["station_type"] == "Inspection")]
    idx = idx[np.argsort(raw.loc[idx, "timestamp"].values)]
    is_running_arr = raw.loc[idx, "is_running"].to_numpy().copy()
    units_arr = raw.loc[idx, "units"].to_numpy().copy()
    scrap_arr = raw.loc[idx, "scrap"].to_numpy().copy()
    cause_q_arr = raw.loc[idx, "cause_Quality"].to_numpy().copy()
    trigger_arr = raw.loc[idx, "trigger_sensor"].to_numpy().copy()
    apply_quality_scrap_rule(is_running_arr, scrap_arr, units_arr, cause_q_arr, trigger_arr)
    raw.loc[idx, "is_running"] = is_running_arr
    raw.loc[idx, "units"] = units_arr
    raw.loc[idx, "scrap"] = scrap_arr
    raw.loc[idx, "cause_Quality"] = cause_q_arr
    raw.loc[idx, "trigger_sensor"] = trigger_arr

raw["good_units"] = raw["units"] - raw["scrap"]
print("Quality (Inspection) eventos tras la regla de 3 scrap consecutivos:", int(raw.loc[raw['station_type']=='Inspection','cause_Quality'].sum()))
# =================== FIN AÑADIDO (regla Quality Inspection) ===================


Quality (Inspection) eventos tras la regla de 3 scrap consecutivos: 11


In [8]:
# OEE definitions (original, sin cambios):
# Availability = running_time / planned_time
# Performance  = actual_output / (ideal_rate * running_time)
# Quality      = good_units / total_units
# OEE          = A * P * Q
IDEAL_RATE = 6

def _oee_from_agg(df, ideal_rate=IDEAL_RATE):
    df = df.copy()
    df["availability"] = np.where(df["planned_min"] > 0,
                                  df["running_min"] / df["planned_min"], 0.0)
    df["performance"]  = np.where(df["running_min"] > 0,
                                  df["total_units"] / (ideal_rate * df["running_min"]), 0.0)
    # AÑADIDO (fix): la produccion sigue una Poisson con media variable
    # (base_rate + pico de inicio de turno), que ocasionalmente supera el
    # ideal_rate=6 por puro azar -> performance>1.0, sobre todo en dias
    # PARCIALES (el primer y ultimo dia de la simulacion no cubren 24h
    # completas, empiezan a las 06:00, asi que tienen menos minutos y mas
    # varianza relativa). Convencion estandar de OEE: performance nunca
    # deberia superar 1.0 (100% de la velocidad ideal) -> se recorta.
    df["performance"] = df["performance"].clip(upper=1.0)
    df["quality"]      = np.where(df["total_units"] > 0,
                                  df["good_units"] / df["total_units"], 0.0)
    df["oee"] = df["availability"] * df["performance"] * df["quality"]
    return df

# MODIFICADO: se añaden station_id/station_type al agrupamiento (antes solo
# machine/day/shift), para poder analizar OEE por estacion mas adelante.
oee_shift = (
    raw.groupby(["machine", "station_id", "station_type", "day", "shift"], as_index=False)
       .agg(planned_min=("is_running", "size"),
            running_min=("is_running", "sum"),
            total_units=("units", "sum"),
            good_units=("good_units", "sum"))
)
oee_shift = _oee_from_agg(oee_shift, ideal_rate=IDEAL_RATE)
oee_shift = oee_shift.sort_values(["machine", "day", "shift"]).reset_index(drop=True)

oee_day = (
    raw.groupby(["machine", "station_id", "station_type", "day"], as_index=False)
       .agg(planned_min=("is_running", "size"),
            running_min=("is_running", "sum"),
            total_units=("units", "sum"),
            good_units=("good_units", "sum"))
)
oee_day = _oee_from_agg(oee_day, ideal_rate=IDEAL_RATE)
oee_day = oee_day.sort_values(["machine", "day"]).reset_index(drop=True)

oee_shift.to_csv(f"{WORK}/oee_by_shift.csv", index=False)
oee_day.to_csv(f"{WORK}/oee_by_day.csv", index=False)
print("Saved:", f"{WORK}/oee_by_shift.csv", "and", f"{WORK}/oee_by_day.csv")
oee_day.head()


Saved: ./output/oee_by_shift.csv and ./output/oee_by_day.csv


,machine,station_id,station_type,day,planned_min,running_min,total_units,good_units,availability,performance,quality,oee
0,M1,ST1,Robotics,2025-01-01,1080,544,3347,3336,0.503704,1.0,0.996713,0.502048
1,M1,ST2,CNC,2025-01-01,1080,537,3339,3328,0.497222,1.0,0.996706,0.495584
2,M1,ST3,Robotics,2025-01-01,1080,542,3346,3330,0.501852,1.0,0.995218,0.499452
3,M1,ST4,Assembly,2025-01-01,1080,545,3353,3342,0.504630,1.0,0.996719,0.502974
4,M1,ST5,Welding,2025-01-01,1080,546,3360,3348,0.505556,1.0,0.996429,0.503750


## Cell 5 — OEE Calculation (Day & Shift)

Compute:
- **Availability** = running_time / planned_time
- **Performance** = actual_output / (ideal_rate * running_time)
- **Quality** = good_output / total_output  
Then **OEE = A * P * Q**.

We summarize by **machine × day** and **machine × day × shift**, and save to CSV.


In [9]:
# Funcion original, sin cambios: convierte flags de inicio de causa en
# intervalos [start, end, duration].
def intervals_from_flags(df, cause_col="cause_Mechanical"):
    starts = df.index[df[cause_col] == 1].tolist()
    intervals = []
    n = len(df)
    # AÑADIDO (fix): antes se caminaba hacia adelante SOLO mientras
    # is_running==0, sin mirar si aparecia el disparo de OTRA causa por el
    # camino. Si dos bloques de causas distintas quedaban pegados sin ningun
    # minuto de por medio (ej. vibration termina justo cuando empieza
    # air_pressure), el evento de la primera causa 'se comia' el bloque de
    # la segunda, generando dos eventos que parecian solaparse. Ahora se
    # corta el avance en cuanto aparece CUALQUIER otro disparo de causa
    # (any_trigger), sea la misma columna u otra distinta.
    all_cause_cols = [c for c in df.columns if c.startswith("cause_")]
    any_trigger = (df[all_cause_cols].sum(axis=1) > 0).to_numpy()
    for s in starts:
        e = min(s + 1, n)
        while e < n and df.loc[e, "is_running"] == 0 and not any_trigger[e]:
            e += 1
        dur = max(0, e - s)
        if dur > 0:
            ts_start = df.loc[s, "timestamp"]
            ts_end   = df.loc[e - 1, "timestamp"]
            trig = df.loc[s, "trigger_sensor"] if "trigger_sensor" in df.columns else ""  # AÑADIDO
            intervals.append((ts_start, ts_end, dur, trig))
    return intervals

# MODIFICADO: el original agrupaba por (machine, day), lo que TRUNCABA los
# eventos de downtime que cruzan la medianoche. Ahora se itera por cada
# combinacion LINEA x ESTACION (filtrando por 'machine' Y 'station_id'
# juntos, ya que con N_LINES>1 el mismo station_id se repite en cada linea)
# y el 'day' se asigna segun el inicio real del evento.
# AÑADIDO: station_id, station_type, stop_reason_detail (Starved en Robotics),
# trigger_sensor (que sensor concreto disparo el evento, dentro de una misma
# 'cause' que puede tener 2-3 sensores posibles -> ej. cause='Mechanical' con
# trigger_sensor='vibration' O 'air_pressure').
rows = []
for line in LINES:
    for m in MACHINES:
        st = MACHINE_TO_STATION[m]
        g = raw[(raw["machine"] == line) & (raw["station_id"] == st["station_id"])].sort_values("timestamp").reset_index(drop=True)
        for cause in DOWNTIME_CAUSES:
            cause_col = f"cause_{cause}"
            if cause_col not in g.columns:
                continue
            detail = "Operator supply gap (infeed not replenished)" if (st["station_type"]=="Robotics" and cause=="Starved") else ""
            for (ts_start, ts_end, minutes, trig) in intervals_from_flags(g, cause_col=cause_col):
                rows.append({
                    "machine": line, "station_id": st["station_id"], "station_type": st["station_type"],
                    "day": ts_start.date(), "cause": cause, "trigger_sensor": trig, "stop_reason_detail": detail,
                    "start": ts_start, "end": ts_end, "minutes": int(minutes)
                })

downtime_cols = ["machine", "station_id", "station_type", "day", "cause", "trigger_sensor", "stop_reason_detail", "start", "end", "minutes"]
downtime = pd.DataFrame(rows, columns=downtime_cols)

if len(downtime) == 0:
    pareto = pd.DataFrame(columns=["machine", "station_id", "day", "cause", "minutes"])
    print("[INFO] No downtime intervals were detected; Pareto will be empty.")
else:
    # MODIFICADO: se añade station_id al groupby (antes solo machine/day/cause);
    # con N_LINES>1 el mismo station_id se repite en cada linea, asi que sin
    # station_id el Pareto mezclaria estaciones de lineas distintas.
    pareto = (
        downtime.groupby(["machine", "station_id", "day", "cause"], as_index=False)["minutes"]
        .sum()
        .sort_values(["machine", "station_id", "day", "minutes"], ascending=[True, True, True, False])
        .reset_index(drop=True)
    )

pareto.to_csv(f"{WORK}/downtime_pareto.csv", index=False)
downtime.to_csv(f"{WORK}/downtime_events.csv", index=False)
print("Saved:", f"{WORK}/downtime_pareto.csv", "and", f"{WORK}/downtime_events.csv")
pareto.head(10)


Saved: ./output/downtime_pareto.csv and ./output/downtime_events.csv


,machine,station_id,day,cause,minutes
0,M1,ST1,2025-01-01,Blocked,368
1,M1,ST1,2025-01-01,Starved,74
2,M1,ST1,2025-01-01,Mechanical,50
3,M1,ST1,2025-01-01,Quality,32
4,M1,ST1,2025-01-01,Electrical,30
5,M1,ST1,2025-01-02,Blocked,697
6,M1,ST1,2025-01-02,Mechanical,101
7,M1,ST1,2025-01-02,Electrical,92
8,M1,ST1,2025-01-02,Starved,54
9,M1,ST1,2025-01-03,Blocked,531


*(Celda duplicada del notebook original que repetía el cálculo de downtime/Pareto — se eliminó porque ya se cubre arriba con la versión corregida. AÑADIDO/MODIFICADO se explica en la celda anterior.)*

In [10]:
# MODIFICADO: se ha eliminado la generacion y guardado de graficos PNG
# (OEE por dia, Pareto por causa) - este notebook ahora solo genera los CSVs.


*(Celda duplicada del notebook original que repetía los mismos gráficos — se eliminó porque ya se cubre arriba con la versión corregida por (machine, station_id).)*

## Cell 8 — Simple SPC: X-bar / R
Create hourly **subgroups** on running minutes and compute:
- **X̄** (mean units/min),
- **s** (std dev),
- **R** (range).

Plot **X̄** over time to spot shifts or instability. Save `spc_xbar_r.csv` for downstream dashboards.


In [11]:
# Cell 8 — Simple SPC: X-bar / R
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "raw" not in globals():
    raise NameError("`raw` DataFrame not found. Run the earlier cells that build `raw` first.")

if not np.issubdtype(raw["timestamp"].dtype, np.datetime64):
    raw = raw.copy()
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], errors="coerce")

ts = raw.copy()
ts["hour"] = ts["timestamp"].dt.floor("h")
run = ts[ts["is_running"] == 1].copy()

# MODIFICADO: group_cols por defecto pasa a ("machine","station_id","hour")
# en vez de ("machine","hour") -> con N_LINES>1 el mismo station_id se
# repite en cada linea, asi que hace falta la combinacion completa.
def spc_groups(df, group_cols=("machine", "station_id", "hour")):
    if df.empty:
        return pd.DataFrame(columns=[*group_cols, "count", "xbar", "s", "R"])
    agg = (
        df.groupby(list(group_cols), as_index=False)["units"]
          .agg(count="count", xbar="mean", s="std", _max="max", _min="min")
    )
    agg["R"] = agg["_max"] - agg["_min"]
    return agg.drop(columns=["_max", "_min"])

spc = spc_groups(run)
out_csv = f"{WORK}/spc_xbar_r.csv"
spc.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# MODIFICADO: se ha eliminado la generacion de graficos X-bar por estacion (PNG). Solo se conserva el CSV.


Saved: ./output/spc_xbar_r.csv


In [12]:
# Save main raw table for users to re-use
raw.to_csv(f"{WORK}/factory_synth_minutely.csv", index=False)

print("Artifacts written to /kaggle/working:")
for f in sorted(os.listdir(WORK)):
    if f.endswith(".csv"):  # MODIFICADO: solo se listan CSVs, ya no se generan PNGs
        print(" -", f)

print("""
Next steps (pick one and publish a v2):
[ ] Add per-cause Availability loss (Six Big Losses mapping)
[ ] Add Performance losses (micro-stops, reduced speed)
[ ] Add Quality detail (defect types, first-pass yield)
[ ] Add shift-level dashboards and targets (A/B/C comparison)
[ ] Export dataset + README as a public CC0 dataset
[ ] Extend SPC: X-bar/R control limits and alarms
""")


Artifacts written to /kaggle/working:
 - downtime_events.csv
 - downtime_pareto.csv
 - factory_synth_minutely.csv
 - oee_by_day.csv
 - oee_by_shift.csv
 - spc_xbar_r.csv

Next steps (pick one and publish a v2):
[ ] Add per-cause Availability loss (Six Big Losses mapping)
[ ] Add Performance losses (micro-stops, reduced speed)
[ ] Add Quality detail (defect types, first-pass yield)
[ ] Add shift-level dashboards and targets (A/B/C comparison)
[ ] Export dataset + README as a public CC0 dataset
[ ] Extend SPC: X-bar/R control limits and alarms



In [13]:
# Cell 9 — Save Artifacts + Next Steps
# Write key tables to /kaggle/working and list what's available.

import os

os.makedirs(WORK, exist_ok=True)

# Save main minutely table if present
if "raw" in globals() and isinstance(raw, pd.DataFrame) and not raw.empty:
    raw_path = f"{WORK}/factory_synth_minutely.csv"
    raw.to_csv(raw_path, index=False)
    print("Saved:", raw_path)
else:
    print("[INFO] 'raw' table not found or empty; skipping save.")

# Helper to report file presence
def _report(path):
    if os.path.exists(path):
        print("✓", os.path.basename(path), "-", os.path.getsize(path), "bytes")
    else:
        print("• (missing)", os.path.basename(path))

expected = [
    f"{WORK}/factory_synth_minutely.csv",
    f"{WORK}/oee_by_day.csv",
    f"{WORK}/oee_by_shift.csv",
    f"{WORK}/downtime_pareto.csv",
    f"{WORK}/spc_xbar_r.csv",
    # plots (may or may not exist depending on earlier cells)
    # examples (per-machine/day filenames vary): we just list any PNGs found
]

print("\nArtifacts in /kaggle/working:")
for p in expected:
    _report(p)

# MODIFICADO: ya no se generan ni se listan graficos PNG, solo CSVs.

# Next steps checklist
print("""
Next steps (pick one for a v2):
[ ] Add per-cause Availability loss (map to Six Big Losses)
[ ] Add Performance losses (micro-stops, reduced speed) and targets
[ ] Add Quality detail (defect taxonomy, first-pass yield)
[ ] Add shift-level dashboard cards and KPIs (A/B/C comparison)
[ ] Publish the CSVs as a CC0 dataset and link it in this notebook
[ ] Extend SPC with control limits and simple alarms
""")


Saved: ./output/factory_synth_minutely.csv

Artifacts in /kaggle/working:
✓ factory_synth_minutely.csv - 55084661 bytes
✓ oee_by_day.csv - 30279 bytes
✓ oee_by_shift.csv - 85073 bytes
✓ downtime_pareto.csv - 37764 bytes
✓ spc_xbar_r.csv - 372753 bytes

Next steps (pick one for a v2):
[ ] Add per-cause Availability loss (map to Six Big Losses)
[ ] Add Performance losses (micro-stops, reduced speed) and targets
[ ] Add Quality detail (defect taxonomy, first-pass yield)
[ ] Add shift-level dashboard cards and KPIs (A/B/C comparison)
[ ] Publish the CSVs as a CC0 dataset and link it in this notebook
[ ] Extend SPC with control limits and simple alarms



## (Optional) Footer — Work With Me
*Need help applying OEE/PdM/SPC to your process or data? I offer a 60-minute assessment and notebook hand-off. DM to collaborate.*


In [14]:
# ================================
# Submission Writer (safe fallback)
# Writes: /kaggle/working/submission.json
# If `submission_dict` is missing, writes a harmless placeholder instead of crashing.
# ================================
import os, json, sys, time
from datetime import datetime

OUT_PATH = f"{WORK}/submission.json"

def _to_builtin(obj):
    """Recursively convert numpy/pandas types to plain Python for JSON."""
    try:
        import numpy as np
        import pandas as pd
    except Exception:
        np = None; pd = None

    if isinstance(obj, dict):
        return {str(_to_builtin(k)): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_builtin(x) for x in obj]
    if np is not None:
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, (np.bool_,)):
            return bool(obj)
        if isinstance(obj, (np.ndarray,)):
            return obj.tolist()
    # pandas scalars
    if "pandas" in sys.modules:
        import pandas as pd
        if isinstance(obj, (pd.Timestamp,)):
            return obj.isoformat()
        if isinstance(obj, (pd.Timedelta,)):
            return obj.total_seconds()
    return obj

def _is_json_serializable(obj) -> bool:
    try:
        json.dumps(obj)
        return True
    except Exception:
        return False

def _validate_submission(d):
    # Minimal sanity checks
    if not isinstance(d, dict):
        print("[WARN] Expected dict for submission, got:", type(d).__name__)
        return
    if len(d) == 0:
        print("[WARN] submission_dict is empty.")

# Build or fallback
if "submission_dict" in globals():
    submission = _to_builtin(submission_dict)
    source = "user-provided submission_dict"
else:
    # Fallback placeholder (this notebook is not a competition notebook)
    submission = {
        "_placeholder": True,
        "_note": "This is a non-competition notebook. Replace with a real submission_dict in a competition context.",
        "notebook": "Factory OEE & Downtime (Synthetic)",
        "generated_at": datetime.utcnow().isoformat() + "Z"
    }
    source = "auto-generated placeholder"

_validate_submission(submission)

# Ensure JSON-serializable
if not _is_json_serializable(submission):
    submission = _to_builtin(submission)
    if not _is_json_serializable(submission):
        raise ValueError("Submission object is not JSON-serializable even after conversion.")

# Write
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(submission, f, separators=(",", ":"), ensure_ascii=False)

# Read-back verification
sz = os.path.getsize(OUT_PATH)
head = open(OUT_PATH, "r").read(400)

print(f"✅ Wrote {OUT_PATH} from {source}")
print("Bytes:", sz)
print("Preview (first 400 chars):")
print(head)

if sz < 2:
    raise IOError("Submission file is too small; likely empty.")

print("\n[INFO] For a real competition:")
print("- Build a dict named `submission_dict` earlier in the notebook that matches the competition's Evaluation schema.")
print("- Keep this cell as the last cell so the file is present under Output -> submission.json")


✅ Wrote ./output/submission.json from auto-generated placeholder
Bytes: 222
Preview (first 400 chars):
{"_placeholder":true,"_note":"This is a non-competition notebook. Replace with a real submission_dict in a competition context.","notebook":"Factory OEE & Downtime (Synthetic)","generated_at":"2026-09-11T06:56:19.165302Z"}

[INFO] For a real competition:
- Build a dict named `submission_dict` earlier in the notebook that matches the competition's Evaluation schema.
- Keep this cell as the last cell so the file is present under Output -> submission.json


C:\Users\alber\AppData\Local\Temp\ipykernel_21036\448642571.py:66: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z"
